# 00 — Extracción y exploración — Exploit.in

**Exploit.in** fue uno de los foros underground rusos más activos entre 2005 y 2008.  
Motor: **Invision Power Board (IPB)**. Idioma predominante: ruso.  
El dump SQL (~190 MB descomprimido) contiene:

| Tabla | Contenido |
|---|---|
| `ibf_posts` | 80,891 mensajes públicos del foro |
| `ibf_message_text` | 14,318 mensajes privados (contenido) |
| `ibf_message_topics` | 14,432 hilos de PM (remitente, destinatario) |
| `ibf_topics` | 13,925 hilos del foro |
| `ibf_members` | 9,647 usuarios registrados |
| `ibf_reputation` | 4,785 votos de reputación con comentarios |
| `ibf_forums` | 41 secciones del foro |

Este notebook extrae todas las tablas, guarda parquets en `data/processed/` y produce  
visualizaciones exploratorias del foro.

## 0. Setup

In [ ]:
# "sys" permite modificar el comportamiento del intérprete de Python,
# en este caso para añadir carpetas adicionales donde buscar módulos.
import sys

# "pandas" es la librería más usada para trabajar con tablas de datos en Python.
# Un "DataFrame" de pandas es como una hoja de Excel que podemos manipular con código.
import pandas as pd

# "matplotlib.pyplot" es la librería estándar para crear gráficas en Python.
# La importamos como "plt" por convención, para escribir menos.
import matplotlib.pyplot as plt

# "matplotlib.ticker" contiene herramientas para personalizar el formato
# de los números en los ejes de las gráficas (por ejemplo, mostrar 10,000 en lugar de 10000).
import matplotlib.ticker as mticker

# "pathlib.Path" es una herramienta para manejar rutas de archivos y carpetas
# de forma más cómoda y segura que con cadenas de texto normales.
from pathlib import Path

# "IPython.display.display" permite mostrar tablas de pandas formateadas
# directamente en la celda de salida del notebook, en lugar de simplemente imprimirlas.
from IPython.display import display

# Añadimos el directorio actual ('.') al principio de la lista de rutas donde
# Python busca módulos. Esto es necesario para poder importar nuestro propio
# módulo 'src.loaders' que está en la carpeta 'src/' de este proyecto.
sys.path.insert(0, '.')

# Importamos la función principal de nuestro parser personalizado.
from src.loaders import load_exploitin

# Definimos la ruta al archivo ZIP que contiene el dump SQL de Exploit.in.
# ".." significa "subir un nivel" en la jerarquía de carpetas.
# Path usa '/' para separar carpetas, independientemente del sistema operativo.
SQL_ZIP   = Path('../../../data_bruto/Hacking Forums/Exploit.in_2013.12.13.zip')

# Definimos dónde guardaremos los archivos procesados (en formato Parquet).
PROCESSED = Path('data/processed')

# Creamos la carpeta si no existe. "parents=True" también crea las carpetas
# intermedias necesarias. "exist_ok=True" evita un error si ya existe.
PROCESSED.mkdir(parents=True, exist_ok=True)

# Verificamos que el archivo ZIP existe antes de continuar.
# Si no existe, "assert" lanzará un error con el mensaje indicado,
# para que sepamos exactamente qué falta.
assert SQL_ZIP.exists(), f'No encontrado: {SQL_ZIP}'

# Mostramos información básica del archivo: nombre y tamaño en megabytes.
# "SQL_ZIP.stat().st_size" devuelve el tamaño en bytes; dividimos entre 1e6 para pasarlo a MB.
print(f'ZIP: {SQL_ZIP.name}  ({SQL_ZIP.stat().st_size/1e6:.0f} MB)')

## 1. Carga y guardado de parquets

In [ ]:
# Llamamos a nuestra función principal del parser.
# Esta función hace todo el trabajo pesado: lee el ZIP, parsea el SQL en dos pasadas,
# limpia el HTML y devuelve un diccionario con todos los DataFrames listos.
tables = load_exploitin(SQL_ZIP)

# Extraemos cada tabla del diccionario en una variable separada
# para facilitar su uso en el resto del notebook.
posts          = tables['posts']           # 80,891 mensajes públicos del foro
topics         = tables['topics']          # 13,925 hilos de discusión
forums         = tables['forums']          # 41 secciones del foro
members        = tables['members']         # 9,647 usuarios registrados
message_text   = tables['message_text']    # 14,318 mensajes privados (contenido)
message_topics = tables['message_topics']  # 14,432 hilos de mensajes privados (metadatos)
reputation     = tables['reputation']      # 4,785 votos de reputación

# Guardamos cada tabla en formato Parquet, que es mucho más eficiente que CSV:
# ocupa menos espacio, se lee más rápido y conserva los tipos de datos (fechas, números, etc.).
# Para la tabla 'forums' y 'members' solo guardamos las columnas más relevantes,
# descartando columnas técnicas del motor del foro (skin, configuración interna, etc.)
save_map = {
    'posts':          posts,
    'topics':         topics,
    # Solo las columnas de foros que tienen sentido para el análisis.
    'forums':         forums[['id','name','description','topics','posts','parent_id']],
    # Solo las columnas de miembros relevantes para el análisis de usuarios.
    'members':        members[['id','name','mgroup','posts','rep','joined','last_visit','language','ip_address']],
    'message_text':   message_text,
    'message_topics': message_topics,
    'reputation':     reputation,
}

# Iteramos sobre el diccionario y guardamos cada DataFrame como un archivo .parquet.
for name, df in save_map.items():
    # Construimos la ruta de salida uniendo la carpeta PROCESSED con el nombre del archivo.
    out = PROCESSED / f'{name}.parquet'

    # Guardamos el DataFrame como Parquet. "index=False" evita que se guarde también
    # el índice de las filas (que normalmente son solo números del 0 al N, sin información útil).
    df.to_parquet(out, index=False)
    print(f'  {name:20s} → {out.name}  ({len(df):,} filas)')

print('\nTodos los parquets guardados.')

## 2. Estructura del foro

In [ ]:
# Filtramos solo las secciones que tienen al menos un post (posts > 0),
# para no mostrar secciones vacías en el gráfico.
# Seleccionamos solo las columnas relevantes: nombre, número de hilos y número de posts.
# Ordenamos de mayor a menor número de posts para que el gráfico tenga sentido.
# reset_index(drop=True) reinicia el índice numérico de 0 en adelante.
active_forums = (
    forums[forums['posts'] > 0][['name','topics','posts']]
    .sort_values('posts', ascending=False)
    .reset_index(drop=True)
)

# Creamos una figura (fig) y un área de dibujo (ax) de 10x7 pulgadas.
# "subplots" es la forma estándar de crear gráficas en matplotlib.
fig, ax = plt.subplots(figsize=(10, 7))

# Creamos un gráfico de barras horizontal (barh).
# Usamos [::-1] para invertir el orden, de modo que la sección con más posts
# quede arriba (matplotlib dibuja de abajo a arriba por defecto).
# "color='#c0392b'" es un rojo oscuro, "alpha=0.85" hace las barras ligeramente transparentes.
bars = ax.barh(active_forums['name'][::-1], active_forums['posts'][::-1],
               color='#c0392b', alpha=0.85)

# Etiqueta del eje X (horizontal).
ax.set_xlabel('Número de posts')

# Título de la gráfica.
ax.set_title('Exploit.in — Actividad por sección del foro')

# Formateamos los números del eje X para mostrar separador de miles (coma en inglés).
# FuncFormatter recibe una función que toma el valor y devuelve el texto a mostrar.
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Añadimos el número exacto de posts al final de cada barra como etiqueta de texto.
# Iteramos sobre las barras y sus valores correspondientes.
for bar, val in zip(bars, active_forums['posts'][::-1]):
    ax.text(bar.get_width() + 30,                    # posición X: justo a la derecha de la barra
            bar.get_y() + bar.get_height()/2,        # posición Y: centrado en la barra
            f'{val:,}',                              # texto: el número formateado con comas
            va='center', fontsize=8)                 # alineación vertical al centro

# Ajustamos automáticamente los márgenes para que nada quede cortado.
plt.tight_layout()
plt.show()

# Mostramos estadísticas en texto.
print(f'\nTotal secciones activas: {len(active_forums)}')
print(f'Total posts en el foro:  {active_forums["posts"].sum():,}')

## 3. Timeline de actividad

In [ ]:
# Filtramos los posts que tienen fecha conocida (eliminamos filas con post_date = NaN/None).
# "dropna(subset=['post_date'])" elimina filas donde 'post_date' sea nulo.
# ".copy()" crea una copia para evitar advertencias de pandas al modificar la tabla.
posts_dated = posts.dropna(subset=['post_date']).copy()

# Creamos una nueva columna 'month' que contiene el mes y año de cada post.
# ".dt" es el accesorio para trabajar con columnas de fecha en pandas.
# ".to_period('M')" convierte una fecha a un "periodo mensual" (ej. 2007-03 para marzo de 2007).
# Esto nos permite agrupar todos los posts de un mismo mes independientemente del día u hora.
posts_dated['month'] = posts_dated['post_date'].dt.to_period('M')

# Contamos cuántos posts hay por mes.
# "groupby('month').size()" agrupa las filas por mes y cuenta cuántas hay en cada grupo.
monthly = posts_dated.groupby('month').size()

# Convertimos los periodos mensuales a marcas de tiempo (timestamps) para poder
# graficarlos correctamente en un eje de tiempo continuo.
monthly.index = monthly.index.to_timestamp()

# Creamos la gráfica: una figura ancha (13 pulgadas) y baja (4 pulgadas).
fig, ax = plt.subplots(figsize=(13, 4))

# "fill_between" rellena el área bajo la curva con color semitransparente (alpha=0.3).
# Esto crea un efecto visual de "área" que facilita ver la tendencia general.
ax.fill_between(monthly.index, monthly.values, alpha=0.3, color='#c0392b')

# Dibujamos también la línea de la curva encima del área, con grosor lw=1.5.
ax.plot(monthly.index, monthly.values, color='#c0392b', lw=1.5)

# Título y etiqueta del eje Y.
ax.set_title('Exploit.in — Posts por mes')
ax.set_ylabel('Posts')

# Formateamos el eje Y para mostrar separadores de miles.
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

# Encontramos el mes con más actividad: "idxmax()" devuelve el índice (mes) del valor máximo.
peak = monthly.idxmax()
print(f'Periodo: {monthly.index.min().strftime("%Y-%m")} → {monthly.index.max().strftime("%Y-%m")}')
print(f'Pico de actividad: {peak.strftime("%Y-%m")} con {monthly.max():,} posts')

## 4. Usuarios más activos y sistema de reputación

In [ ]:
# Obtenemos el top 20 de usuarios por número total de posts publicados.
# Primero filtramos los que tienen al menos 1 post (descartamos cuentas inactivas).
# Seleccionamos las columnas relevantes y ordenamos de mayor a menor.
# "reset_index(drop=True)" renueva el índice numérico para que empiece en 0.
top_posters = (
    members[members['posts'] > 0][['name','posts','rep','joined']]
    .sort_values('posts', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
# Hacemos que el índice empiece en 1 (más natural para el lector que empezar en 0).
top_posters.index += 1
# Renombramos las columnas para que el texto de salida sea más legible.
top_posters.columns = ['Usuario', 'Posts', 'Reputación', 'Alta']

print('Top 20 usuarios por posts:')
display(top_posters)

# Obtenemos el top 20 de usuarios por puntos de reputación acumulados.
# La reputación refleja la confianza de la comunidad en ese usuario.
top_rep = (
    members[members['rep'] > 0][['name','rep','posts']]
    .sort_values('rep', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top_rep.index += 1

# Creamos una figura con dos gráficas lado a lado (1 fila, 2 columnas).
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica izquierda: top 20 usuarios por posts totales (azul).
axes[0].barh(top_posters['Usuario'][::-1], top_posters['Posts'][::-1], color='#2980b9', alpha=0.85)
axes[0].set_title('Top 20 — Posts totales')
axes[0].set_xlabel('Posts')

# Gráfica derecha: top 20 usuarios por puntos de reputación (verde).
axes[1].barh(top_rep['name'][::-1], top_rep['rep'][::-1], color='#27ae60', alpha=0.85)
axes[1].set_title('Top 20 — Reputación')
axes[1].set_xlabel('Puntos de reputación')

plt.tight_layout()
plt.show()

## 5. Distribución de idiomas de usuarios

In [ ]:
# Contamos cuántos usuarios tienen cada idioma configurado en su perfil.
# ".value_counts()" devuelve los valores únicos ordenados de más a menos frecuente.
# ".head(10)" limita a los 10 idiomas más comunes.
lang_counts = members['language'].value_counts().head(10)
print('Idiomas configurados por los usuarios:')
# ".to_string()" evita que pandas trunque la salida si hay muchas filas.
print(lang_counts.to_string())

# Mostramos una muestra aleatoria de posts en ruso para confirmar el idioma mayoritario.
# Filtramos posts con más de 80 caracteres para evitar mensajes muy cortos que no son representativos.
print('\nMuestra de posts en ruso (primeros 3 con contenido > 80 chars):')
sample_posts = (
    posts[posts['content'].str.len() > 80]
    # ".sample(3, random_state=42)" toma 3 filas al azar.
    # "random_state=42" fija la "semilla" del generador aleatorio para que
    # la muestra sea siempre la misma al re-ejecutar la celda (reproducibilidad).
    .sample(3, random_state=42)[['author_name','post_date','content']]
)
# Iteramos sobre las filas de la muestra para mostrar cada post formateado.
# ".iterrows()" devuelve pares (índice, fila) que podemos desempaquetar con "_" y "row".
for _, row in sample_posts.iterrows():
    # Formateamos la fecha como "AAAA-MM-DD" usando strftime.
    print(f'\n[{row["author_name"]} · {row["post_date"].strftime("%Y-%m-%d")}]')
    # Mostramos solo los primeros 300 caracteres para no saturar la pantalla.
    print(row['content'][:300])

## 6. Posts por sección — Secciones clave para análisis

In [ ]:
# Construimos dos diccionarios de "traducción" para poder enriquecer los posts con
# información de otras tablas (equivalente a JOIN en SQL).

# Mapa: id de foro → nombre del foro
forum_name_map = dict(zip(forums['id'], forums['name']))

# Mapa: id de hilo → id del foro al que pertenece ese hilo
topic_forum_map = dict(zip(topics['tid'], topics['forum_id']))

# Añadimos a cada post el id del foro al que pertenece, consultando
# primero en qué foro está su hilo (topic_id → forum_id).
# ".map()" aplica el diccionario como tabla de traducción a cada valor de la columna.
posts['forum_id']   = posts['topic_id'].map(topic_forum_map)

# Luego convertimos el id del foro en su nombre legible.
posts['forum_name'] = posts['forum_id'].map(forum_name_map)

# Lista de secciones del foro que tienen mayor interés para el análisis
# de inteligencia de amenazas (threat intelligence).
# Incluye secciones sobre hacking, dinero, marketplace, malware, etc.
ti_sections = [
    'Безопасность и взлом',            # Seguridad y hacking
    'Деньги',                           # Dinero
    'Покупка/Продажа/Обмен/Работа',     # Compra/Venta/Intercambio/Trabajo
    'Black List',                        # Lista negra de estafadores
    'Программирование',                 # Programación
    'Вирусология',                      # Virología/Malware
    'Криптография и приватность',       # Criptografía y privacidad
    'Спам, рассылки',                   # Spam y correo masivo
    '1st Access Level',                 # Sección premium nivel 1
    '2nd Access Level',                 # Sección premium nivel 2
]

# Contamos los posts agrupados por sección del foro y ordenamos de más a menos.
posts_by_section = posts.groupby('forum_name').size().sort_values(ascending=False)

print('Posts por sección (todas):')
print(posts_by_section.to_string())

# Filtramos para mostrar solo las secciones de interés.
# ".reindex()" selecciona solo las secciones de la lista; si alguna no existe,
# pone NaN. Filtramos primero para asegurarnos de que existan.
print('\nPosts en secciones de interés TI:')
ti_counts = posts_by_section.reindex([s for s in ti_sections if s in posts_by_section.index])
print(ti_counts.to_string())

# Calculamos qué porcentaje del total representan las secciones de interés.
print(f'\nTotal posts en secciones TI: {ti_counts.sum():,} de {len(posts):,} totales ({ti_counts.sum()/len(posts)*100:.1f}%)')

## 7. Mensajes privados — Red de comunicación

In [ ]:
# Filtramos los mensajes privados para quedarnos solo con los mensajes "reales":
# - Descartamos los que no tienen nombre de remitente (from_name) ni destinatario (to_name).
#   .notna() devuelve True si el valor NO es nulo.
# - Excluimos los mensajes enviados por 'toha' (el administrador), porque son
#   mensajes automáticos de bienvenida a nuevos usuarios, no mensajes reales.
# ".copy()" crea una copia independiente del subconjunto filtrado.
pms = message_topics[
    message_topics['from_name'].notna() &
    message_topics['to_name'].notna() &
    (message_topics['from_name'] != 'toha')  # excluye mensajes automáticos del admin
].copy()

# Contamos cuántos mensajes privados envió cada usuario y nos quedamos con los 15 primeros.
# ".value_counts()" cuenta las ocurrencias de cada valor único en la columna 'from_name'.
pm_senders = pms['from_name'].value_counts().head(15)

# Creamos una gráfica de barras horizontal para visualizar los top emisores de PMs.
fig, ax = plt.subplots(figsize=(10, 5))
# "[::-1]" invierte el orden para que el que más envió quede arriba.
# ".plot.barh()" es una forma abreviada de crear barras horizontales en pandas.
pm_senders[::-1].plot.barh(ax=ax, color='#8e44ad', alpha=0.85)
ax.set_title('Exploit.in — Top 15 emisores de mensajes privados')
ax.set_xlabel('PMs enviados')
plt.tight_layout()
plt.show()

# Mostramos estadísticas resumidas de la red de mensajes privados.
print(f'Total PMs directos (excl. bienvenidas): {len(pms):,}')
# ".nunique()" cuenta los valores únicos (número de usuarios distintos).
print(f'Usuarios únicos que enviaron PMs: {pms["from_name"].nunique():,}')
print(f'Usuarios únicos que recibieron PMs: {pms["to_name"].nunique():,}')

## 8. Sistema de reputación — Comentarios

In [ ]:
# Creamos un diccionario para traducir IDs de usuario a nombres legibles.
# "dict(zip(col_a, col_b))" es una forma eficiente de crear un diccionario
# emparejando dos columnas: {id_1: nombre_1, id_2: nombre_2, ...}
member_names = dict(zip(members['id'], members['name']))

# Añadimos los nombres del dador y receptor de cada voto de reputación.
# ".map(member_names)" reemplaza cada ID por el nombre correspondiente del diccionario.
reputation['giver']    = reputation['from_id'].map(member_names)
reputation['receiver'] = reputation['member_id'].map(member_names)

# Contamos los votos positivos (CODE='01') y negativos (CODE='02').
# "==" compara elemento a elemento y devuelve una Serie de True/False.
# ".sum()" cuenta cuántos True hay (True se suma como 1, False como 0).
rep_pos = (reputation['CODE'] == '01').sum()  # positivo
rep_neg = (reputation['CODE'] == '02').sum()  # negativo

# Contamos cuántos votos tienen un comentario de texto adicional.
rep_with_comment = reputation['message'].notna().sum()

print(f'Votos de reputación totales: {len(reputation):,}')
print(f'  Positivos (01): {rep_pos:,}')
print(f'  Negativos (02): {rep_neg:,}')
print(f'  Con comentario: {rep_with_comment:,} ({rep_with_comment/len(reputation)*100:.0f}%)')

# Mostramos una muestra aleatoria de 5 comentarios de reputación.
print('\nMuestra de comentarios de reputación:')
sample_rep = reputation[reputation['message'].notna()].sample(5, random_state=42)
for _, row in sample_rep.iterrows():
    # Mostramos un triángulo hacia arriba ▲ para positivo, hacia abajo ▼ para negativo.
    code = '▲' if row['CODE'] == '01' else '▼'
    # Formateamos: símbolo · dador → receptor · primeros 80 caracteres del comentario.
    # str(...) convierte a texto por si hay valores nulos.
    print(f"  {code} {row['giver']:15s} → {row['receiver']:15s}: {str(row['message'])[:80]}")

## 9. Muestra de posts por sección clave

In [ ]:
# Lista de secciones de las que queremos ver una muestra de posts.
sections_to_sample = [
    'Безопасность и взлом',            # Seguridad y hacking
    'Деньги',                           # Dinero
    'Покупка/Продажа/Обмен/Работа',     # Compra/Venta/Intercambio/Trabajo
    'Вирусология',                      # Virología/Malware
]

# Iteramos sobre cada sección para mostrar una muestra de sus posts.
for section in sections_to_sample:
    # Filtramos los posts de esta sección con contenido de más de 60 caracteres.
    # Posts muy cortos (menos de 60 chars) suelen ser respuestas breves sin contenido útil.
    section_posts = posts[
        (posts['forum_name'] == section) &
        (posts['content'].str.len() > 60)
    ]

    # Si no hay posts para esta sección, la saltamos.
    if len(section_posts) == 0:
        continue

    # Tomamos una muestra de 2 posts al azar. "min(2, len(...))" garantiza que
    # no pedimos más posts de los que hay disponibles.
    sample = section_posts.sample(min(2, len(section_posts)), random_state=42)

    # Imprimimos un separador visual y el nombre de la sección.
    print(f'\n{'='*60}')
    print(f'Sección: {section}')
    print('='*60)

    # Mostramos cada post de la muestra con autor, fecha y texto (primeros 350 chars).
    for _, row in sample.iterrows():
        # Comprobamos si la fecha no es nula antes de formatearla.
        # "pd.notna()" devuelve True si el valor no es nulo.
        date_str = row['post_date'].strftime('%Y-%m-%d') if pd.notna(row['post_date']) else 'N/A'
        print(f'[{row["author_name"]} · {date_str}]')
        print(row['content'][:350])
        print()

## 10. Resumen del dataset

In [ ]:
# Imprimimos un resumen completo del dataset cargado con todas las cifras clave.
print('='*55)
print('EXPLOIT.IN — RESUMEN DEL DATASET')
print('='*55)

# Mostramos el rango de fechas del periodo activo del foro.
# ".min()" y ".max()" obtienen la fecha más antigua y más reciente respectivamente.
# ".strftime('%Y-%m')" formatea la fecha como "año-mes" (ej. "2005-02").
print(f'Periodo activo : {posts_dated["post_date"].min().strftime("%Y-%m")} → {posts_dated["post_date"].max().strftime("%Y-%m")}')
print(f'Posts públicos : {len(posts):,}')
print(f'Hilos          : {len(topics):,}')
print(f'Secciones      : {len(forums):,} ({len(active_forums)} activas)')
print(f'Miembros       : {len(members):,}')
print(f'Mensajes priv. : {len(message_text):,}')
print(f'Votos rep.     : {len(reputation):,}')
print()

# Listamos todos los archivos Parquet que se generaron en la carpeta de datos procesados.
# "PROCESSED.glob('*.parquet')" busca todos los archivos .parquet en esa carpeta.
# "sorted(...)" los ordena alfabéticamente para que la lista sea consistente.
print('Parquets generados en data/processed/:')
for f in sorted(PROCESSED.glob('*.parquet')):
    print(f'  {f.name}')